# Density compensation in the ocean

Temperature and salinity differences can offset one another in seawater density. This notebook evaluates the **model minus WOA** density difference at 10, 100, 500, 1000 and 2000 m and attributes it to potential temperature and practical salinity with TEOS-10. The model supplies 1981–2010 annual fields; WOA supplies one already averaged `decav81B0` climate-normal field for that period. The symmetric attribution closes exactly even though the equation of state is nonlinear.

The notebook is standalone and imports no local project files. It loads real CMIP6 and OBS6 data through ESMValCore on the configured Gadi archive. The default WOA facets match the existing Gadi WOA18 OBS6 files (`version=2018`, representative year 2000); verify that both `thetao` and `so` came from the `decav81B0` 1981–2010 normal. Check the model, ensemble and grid in the first code cell. Missing files raise an error; no example fields are substituted. ACCESS-OM3 needs a verified conversion of its model output to potential temperature and practical salinity before applying this CMIP-style comparison.

In [ ]:
from pathlib import Path
import io
import os

import gsw
import iris
import iris.analysis.cartography
import iris.coord_systems
import iris.cube
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

MODEL = 'ACCESS-CM2'
ENSEMBLE = 'r1i1p1f1'
GRID = 'gn'
HISTORICAL_YEARS = '1981/2010'
WOA_VERSION = '2018'
WOA_TIMERANGE = '2000/2000'  # Representative OBS6 timestamp, not the normal's period.
LEVELS = np.array([10., 100., 500., 1000., 2000.])
TARGET_GRID = '1x1'
SAVE_FIGURES = False
FIGURE_DIR = Path.cwd() / 'density_compensation_figures'
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def show_figure(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=140, bbox_inches='tight')
    display(Image(data=buffer.getvalue()))

print(f'Real data: {MODEL} historical {HISTORICAL_YEARS}, '
      f'WOA18 {WOA_VERSION} (OBS6 timestamp {WOA_TIMERANGE})')

In [ ]:
# Embedded at build time from cosima_common.py and
# hydrographic_benchmark.py. The notebook imports neither file.

EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


def lat_2d(cube):
    """2D latitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("latitude")
    if coord.ndim == 2:
        return coord.points
    lon = cube.coord("longitude").points
    return np.broadcast_to(coord.points[:, None], (coord.shape[0], lon.size))


def lon_2d(cube):
    """2D longitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("longitude")
    if coord.ndim == 2:
        return coord.points
    lat = cube.coord("latitude").points
    return np.broadcast_to(coord.points[None, :], (lat.size, coord.shape[0]))


def guess_bounds(cube, coords=("latitude", "longitude")):
    """Add bounds in place where they are missing."""
    for name in coords:
        if cube.coords(name) and not cube.coord(name).has_bounds():
            try:
                cube.coord(name).guess_bounds()
            except ValueError:  # scalar or single-point coordinate
                pass
    return cube


def horizontal_slice(cube):
    """First slice of a cube over its two horizontal dimensions.

    ``cube.slices(["latitude", "longitude"])`` cannot be used here: on a
    curvilinear grid the two coordinates share both dimensions and iris
    rejects them as non-orthogonal.  Slicing by dimension index works
    for regular and curvilinear grids alike.
    """
    horizontal_dims = sorted(
        set(cube.coord_dims("latitude")) | set(cube.coord_dims("longitude")))
    if cube.ndim == len(horizontal_dims):
        return cube
    index = tuple(slice(None) if dim in horizontal_dims else 0
                  for dim in range(cube.ndim))
    return cube[index]


def spherical_polygon_area(lon_vertices, lat_vertices, radius=EARTH_RADIUS):
    """Area in m2 of spherical polygons given their corner coordinates.

    ``lon_vertices`` and ``lat_vertices`` are in degrees with the corners
    along the last axis, which is the ``(..., 4)`` layout CMOR uses for
    ``vertices_longitude`` / ``vertices_latitude``.  Any number of
    corners works as long as they are ordered around the cell.

    The area is the spherical excess, accumulated over a triangle fan
    from the first corner, using the Van Oosterom & Strackee (1983)
    form:

        tan(E/2) = |v1 . (v2 x v3)|
                   / (1 + v1.v2 + v2.v3 + v3.v1)

    with ``v`` the unit vectors of the corners.  This is *exact* for
    great-circle edges and therefore invariant under rotation of the
    sphere, which matters because a tripolar grid is exactly a rotated
    and stretched one: a formula that is only correct for cells aligned
    with latitude circles would degrade precisely where the grid stops
    being aligned.

    The obvious alternative, the spherical shoelace formula
    ``R^2/2 sum (dlon)(sin lat_i + sin lat_{i+1})``, treats each edge as
    linear in ``(lon, sin lat)`` rather than as a great circle.  It is
    exact for a latitude-longitude box and accurate to second order for
    small cells, but it is not rotation invariant and it fails badly for
    cells spanning large longitude ranges, such as those beside a
    displaced pole.

    The two disagree by about 2e-4 relative on a 2-degree cell, and
    neither is "wrong": they assume different cell edges.  A cell whose
    north and south edges follow *parallels* has the shoelace area,
    which is what ``iris.analysis.cartography.area_weights`` returns; a
    cell with great-circle edges has this one.  Both tile the sphere
    exactly.  The discrepancy is far smaller than the difference
    between either and the model's own ``areacello``, which is why
    ``areacello`` is still preferred when it is published.
    """
    lon = np.deg2rad(np.asarray(lon_vertices, dtype=float))
    lat = np.deg2rad(np.asarray(lat_vertices, dtype=float))
    unit = np.stack([np.cos(lat) * np.cos(lon),
                     np.cos(lat) * np.sin(lon),
                     np.sin(lat)], axis=-1)

    first = unit[..., 0, :]
    excess = np.zeros(unit.shape[:-2])
    for corner in range(1, unit.shape[-2] - 1):
        second = unit[..., corner, :]
        third = unit[..., corner + 1, :]
        triple = np.abs(np.einsum("...i,...i->...",
                                  first, np.cross(second, third)))
        denominator = (1.0
                       + np.einsum("...i,...i->...", first, second)
                       + np.einsum("...i,...i->...", second, third)
                       + np.einsum("...i,...i->...", third, first))
        excess += 2.0 * np.arctan2(triple, denominator)
    return excess * radius**2


def area_from_bounds(cube, radius=EARTH_RADIUS):
    """Cell area in m2 from 2D coordinate bounds, or ``None``.

    This is what makes a tripolar grid workable without ``areacello``.
    CMOR requires bounds on ``latitude`` and ``longitude``, and on a
    curvilinear grid those bounds are the four corner vertices of every
    cell, so the area follows exactly from
    :func:`spherical_polygon_area`.

    ESMValCore does not do this: ``try_adding_calculated_cell_area``
    handles regular and rotated-pole grids and raises
    ``CoordinateMultiDimError`` for everything else.  The information is
    in the file regardless.

    Returns ``None`` when the cube has no usable 2D bounds, so callers
    can fall back.
    """
    horizontal = horizontal_slice(cube)
    lat_coord = horizontal.coord("latitude")
    lon_coord = horizontal.coord("longitude")
    if lat_coord.ndim != 2 or lon_coord.ndim != 2:
        return None
    if lat_coord.bounds is None or lon_coord.bounds is None:
        return None
    if lat_coord.bounds.shape[-1] < 3:
        return None
    return np.ma.masked_invalid(
        spherical_polygon_area(lon_coord.bounds, lat_coord.bounds,
                               radius=radius))


def cell_area(cube):
    """Cell area in m2, preferring the model's own ``areacello``.

    Order of preference:

    1. an ``areacello`` cell measure attached to the cube, which is what
       ESMValCore builds from a ``supplementary_variables`` entry.  This
       is the model's own area and is always the best answer: it
       accounts for the actual grid generation, including any partial
       cells at the coast.
    2. a spherical polygon area from the 2D corner vertices, via
       :func:`area_from_bounds`.  CMOR mandates bounds on latitude and
       longitude, so this works on a tripolar grid with no
       ``areacello`` at all.  It assumes great-circle cell edges on a
       sphere, so it differs from the model's own areas by well under a
       percent for an ocean grid.
    3. spherical areas from 1D latitude/longitude bounds, for a regular
       grid.

    Only if all three fail does this raise, and then with instructions.
    """
    horizontal = horizontal_slice(cube)

    for measure in horizontal.cell_measures():
        if measure.var_name in ("areacello", "areacella", "areacell"):
            return np.ma.masked_invalid(
                np.ma.asarray(
                    horizontal.cell_measure(measure.name()).core_data(),
                    dtype=float).reshape(horizontal.shape))

    from_bounds = area_from_bounds(horizontal)
    if from_bounds is not None:
        return from_bounds

    if horizontal.coord("latitude").ndim > 1:
        raise ValueError(
            "this cube has 2D (curvilinear) latitude/longitude and no cell "
            "corner bounds, so no cell area can be obtained. Either attach "
            "areacello -- in a recipe with\n"
            "    supplementary_variables:\n"
            "      - short_name: areacello\n"
            "        mip: Ofx\n"
            "(add 'exp: piControl' if the experiment you want does not "
            "publish it), in a notebook with "
            "dataset.add_supplementary(short_name='areacello', mip='Ofx') "
            "-- or keep the coordinate bounds, which CMOR requires and "
            "which are enough to compute the area exactly.")

    # iris defaults to an Earth radius of 6367470 m where the cube has
    # no coordinate system, while cell_lengths uses the mean radius of
    # 6371000 m. That is a 0.11% difference in area, small but free to
    # avoid: pin the same sphere so lengths and areas agree.
    horizontal = horizontal.copy()
    guess_bounds(horizontal)
    if horizontal.coord("latitude").coord_system is None:
        sphere = iris.coord_systems.GeogCS(EARTH_RADIUS)
        horizontal.coord("latitude").coord_system = sphere
        horizontal.coord("longitude").coord_system = sphere
    return iris.analysis.cartography.area_weights(horizontal)


REGIONS = (
    ("Global", -90.0, 90.0),
    ("Southern extratropics", -90.0, -30.0),
    ("Tropics", -30.0, 30.0),
    ("Northern extratropics", 30.0, 90.0),
)


REFERENCE_PERIOD = (1981, 2010)


def _annual_fields(cube):
    """Return annual fields as year, depth, latitude, longitude."""
    data = masked_data(cube)
    depth_name = depth_coord_name(cube)
    if depth_name is None:
        raise ValueError("input has no depth coordinate")
    depth = cube.coord(depth_name).points.astype(float)
    depth_axis = cube.coord_dims(depth_name)[0]
    latitude_axis = cube.coord_dims("latitude")[0]
    longitude_axis = cube.coord_dims("longitude")[-1]
    if cube.coords("time") and cube.coord_dims("time"):
        time_axis = cube.coord_dims("time")[0]
        order = (time_axis, depth_axis, latitude_axis, longitude_axis)
        years = np.array([date.year for date in cube.coord("time").units.num2date(
            cube.coord("time").points)], dtype=float)
    else:
        order = (depth_axis, latitude_axis, longitude_axis)
        years = np.array([0.0])
    if len(set(order)) != len(order) or data.ndim != len(order):
        raise ValueError("expected time/depth/latitude/longitude input")
    data = np.transpose(data, order)
    if len(order) == 3:
        data = data[None, ...]
    if years.size != np.unique(years).size:
        raise ValueError("input must have one annual sample per year")
    return data, depth, years


def _compatible(model_cube, reference_cube):
    """Reject silently misaligned depth levels or regridding."""
    for name in ("depth", "latitude", "longitude"):
        model_name = depth_coord_name(model_cube) if name == "depth" else name
        ref_name = depth_coord_name(reference_cube) if name == "depth" else name
        model_points = model_cube.coord(model_name).points
        ref_points = reference_cube.coord(ref_name).points
        if model_points.shape != ref_points.shape or not np.allclose(
                model_points, ref_points, atol=1e-5):
            raise ValueError(f"model and reference {name} grids differ")


def _woa_potential_temperature(temperature_cube, salinity_cube):
    """Convert WOA in-situ temperature to potential temperature at 0 dbar.

    WOA `t_an` is in-situ temperature even when ESMValTool's WOA CMORizer
    labels the field `thetao`. Use its practical salinity, location, and
    pressure to perform the TEOS-10 conversion before comparison.
    """
    _compatible(temperature_cube, salinity_cube)
    temperature_cube = temperature_cube.copy()
    salinity_cube = salinity_cube.copy()
    temperature_cube.convert_units("degC")
    salinity_cube.convert_units("1e-3")
    temperature, depth, _ = _annual_fields(temperature_cube)
    salinity, _, _ = _annual_fields(salinity_cube)
    if temperature.shape != salinity.shape:
        raise ValueError("WOA temperature and salinity shapes differ")
    latitude = lat_2d(temperature_cube)
    longitude = lon_2d(temperature_cube)
    pressure = gsw.p_from_z(-depth[:, None, None], latitude[None, :, :])
    valid = (~np.ma.getmaskarray(temperature)
             & ~np.ma.getmaskarray(salinity)
             & np.isfinite(np.ma.filled(temperature, np.nan))
             & np.isfinite(np.ma.filled(salinity, np.nan)))
    practical = np.ma.filled(salinity, np.nan)
    in_situ = np.ma.filled(temperature, np.nan)
    absolute = gsw.SA_from_SP(practical, pressure[None, ...],
                              longitude[None, None, ...],
                              latitude[None, None, ...])
    potential = gsw.pt0_from_t(absolute, in_situ, pressure[None, ...])
    return np.ma.masked_where(~valid | ~np.isfinite(potential), potential)


def _check_reference_period(years, period=REFERENCE_PERIOD):
    """Require every annual historical field in the WOA normal interval."""
    expected = np.arange(period[0], period[1] + 1)
    if years.shape != expected.shape or not np.array_equal(years, expected):
        raise ValueError(f"historical years must match WOA normal {period}")


def _region_weights(cube):
    area = np.ma.filled(cell_area(cube), 0.0)
    latitude = lat_2d(cube)
    weights = []
    for _, south, north in REGIONS:
        inside = (latitude >= south) & (latitude < north if north < 90
                                         else latitude <= north)
        weights.append(np.where(inside, area, 0.0))
    return np.asarray(weights)


def _weighted_mean(data, weights, valid):
    weight = np.where(valid, weights, 0.0)
    denominator = weight.sum()
    if denominator <= 0:
        return np.nan
    return float(np.sum(np.where(valid, data, 0.0) * weight) / denominator)


def _reference_metrics(historical, reference, weights):
    """Area-weighted mean difference, RMSE and paired-cell coverage."""
    n_region, n_depth = weights.shape[0], historical.shape[1]
    bias = np.full((n_region, n_depth), np.nan)
    rmse = bias.copy()
    coverage = bias.copy()
    model_clim = np.ma.mean(historical, axis=0)
    ref_clim = np.ma.mean(reference, axis=0)
    for region in range(n_region):
        for level in range(n_depth):
            delta = model_clim[level] - ref_clim[level]
            valid = ~np.ma.getmaskarray(delta) & np.isfinite(
                np.ma.filled(delta, np.nan))
            reference_valid = ~np.ma.getmaskarray(ref_clim[level]) & np.isfinite(
                np.ma.filled(ref_clim[level], np.nan))
            total = np.where(reference_valid, weights[region], 0.0).sum()
            if total <= 0:
                continue
            paired = np.where(valid, weights[region], 0.0).sum()
            coverage[region, level] = paired / total
            values = np.ma.filled(delta, 0.0)
            bias[region, level] = _weighted_mean(values, weights[region], valid)
            squared = _weighted_mean(values**2, weights[region], valid)
            rmse[region, level] = np.sqrt(squared) if np.isfinite(squared) else np.nan
    return bias, rmse, coverage


def _control_metrics(control, years, weights):
    """Common-mask area means and least-squares slopes per century."""
    n_time, n_depth = control.shape[:2]
    n_region = weights.shape[0]
    means = np.full((n_time, n_region, n_depth), np.nan)
    coverage = np.full((n_region, n_depth), np.nan)
    slope = coverage.copy()
    if n_time < 2:
        raise ValueError("piControl drift needs at least two annual means")
    for region in range(n_region):
        for level in range(n_depth):
            field = control[:, level]
            common = np.all(~np.ma.getmaskarray(field) & np.isfinite(
                np.ma.filled(field, np.nan)), axis=0)
            first_valid = ~np.ma.getmaskarray(field[0]) & np.isfinite(
                np.ma.filled(field[0], np.nan))
            total = np.where(first_valid, weights[region], 0.0).sum()
            if total <= 0:
                continue
            coverage[region, level] = np.where(
                common, weights[region], 0.0).sum() / total
            for time in range(n_time):
                means[time, region, level] = _weighted_mean(
                    np.ma.filled(field[time], 0.0), weights[region], common)
            if np.isfinite(means[:, region, level]).all():
                slope[region, level] = np.polyfit(
                    years - years[0], means[:, region, level], 1)[0] * 100.0
    anomaly = means - means[0:1]
    return means, anomaly, slope, coverage


def _global_mean_metrics(cube):
    """Mean, first-year anomaly and slope of archived CMIP6 `thetaoga`."""
    time = cube.coord("time")
    values = masked_data(cube)
    if values.ndim != 1 or cube.coord_dims(time) != (0,):
        raise ValueError("thetaoga must be an annual one-dimensional series")
    years = np.array([date.year for date in time.units.num2date(time.points)],
                     dtype=float)
    if len(years) < 2 or len(np.unique(years)) != len(years):
        raise ValueError("thetaoga needs at least two unique annual means")
    if np.ma.getmaskarray(values).any() or not np.isfinite(values).all():
        raise ValueError("thetaoga contains missing annual means")
    anomaly = values - values[0]
    slope = float(np.polyfit(years - years[0], values, 1)[0] * 100.0)
    return years, values, anomaly, slope


def density_components(model_t, model_s, reference_t, reference_s,
                       depth, latitude, longitude):
    """Return exact T/S contributions to model-minus-reference density.

    Four matched three-dimensional fields have shape (depth, y, x).
    TEOS-10 calculates in-situ density at the pressure of each depth.
    The two contributions are the symmetric average over the two possible
    replacement orders, so their sum equals the total density difference
    even with a nonlinear equation of state.
    """
    fields = [np.ma.asarray(field, dtype=float) for field in
              (model_t, model_s, reference_t, reference_s)]
    if len({field.shape for field in fields}) != 1 or fields[0].ndim != 3:
        raise ValueError("T and S fields must share (depth, y, x) shape")
    depth = np.asarray(depth, dtype=float)
    latitude = np.asarray(latitude, dtype=float)
    longitude = np.asarray(longitude, dtype=float)
    if (depth.size != fields[0].shape[0]
            or latitude.shape != fields[0].shape[1:]
            or longitude.shape != latitude.shape):
        raise ValueError("depth and geographic coordinates do not match fields")
    valid = np.logical_and.reduce([
        ~np.ma.getmaskarray(field) & np.isfinite(np.ma.filled(field, np.nan))
        for field in fields])
    pressure = gsw.p_from_z(-depth[:, None, None], latitude[None, :, :])

    def density(potential, practical):
        absolute = gsw.SA_from_SP(np.ma.filled(practical, np.nan), pressure,
                                  longitude[None, :, :],
                                  latitude[None, :, :])
        conservative = gsw.CT_from_pt(absolute,
                                      np.ma.filled(potential, np.nan))
        return gsw.rho(absolute, conservative, pressure)

    model = density(fields[0], fields[1])
    reference = density(fields[2], fields[3])
    warm_only = density(fields[0], fields[3])
    salty_only = density(fields[2], fields[1])
    total = model - reference
    thermal = 0.5 * ((warm_only - reference) + (model - salty_only))
    haline = 0.5 * ((salty_only - reference) + (model - warm_only))
    mask = ~valid | ~np.isfinite(total) | ~np.isfinite(thermal) \
        | ~np.isfinite(haline)
    return tuple(np.ma.masked_where(mask, value)
                 for value in (thermal, haline, total))


def density_metrics(thermal, haline, total, reference_valid, weights):
    """Return regional means, RMSE, coverage and cancellation fraction.

    Cancellation is the area-and-effect-weighted fraction of the two
    absolute density contributions that offsets locally. Zero means
    reinforcement; one means perfect cancellation. Cells with tiny
    contributions have correspondingly tiny influence on this ratio.
    """
    n_region, n_depth = weights.shape[0], total.shape[0]
    result = {name: np.full((n_region, n_depth), np.nan)
              for name in ("thermal", "haline", "total", "rmse",
                           "coverage", "cancellation")}
    for region in range(n_region):
        for level in range(n_depth):
            weight = np.asarray(weights[region], dtype=float)
            paired = (~np.ma.getmaskarray(total[level])
                      & np.isfinite(np.ma.filled(total[level], np.nan)))
            ref = np.asarray(reference_valid[level], dtype=bool)
            possible = np.where(ref, weight, 0.0).sum()
            area = np.where(paired, weight, 0.0)
            used = area.sum()
            if possible <= 0:
                continue
            result["coverage"][region, level] = used / possible
            if used <= 0:
                continue
            for name, field in (("thermal", thermal), ("haline", haline),
                                ("total", total)):
                result[name][region, level] = (
                    np.where(paired, np.ma.filled(field[level], 0.0), 0.0)
                    * area).sum() / used
            values = np.ma.filled(total[level], 0.0)
            result["rmse"][region, level] = np.sqrt(
                (area * values**2).sum() / used)
            t = np.ma.filled(thermal[level], 0.0)
            s = np.ma.filled(haline[level], 0.0)
            magnitude = (area * (np.abs(t) + np.abs(s))).sum()
            if magnitude > 0:
                cancelled = (area * (np.abs(t) + np.abs(s)
                                     - np.abs(values))).sum()
                result["cancellation"][region, level] = (
                    np.clip(cancelled / magnitude, 0.0, 1.0))
    return result

## Prepare matched hydrographic fields

The notebook starts a Dask distributed client before ESMValCore loads and preprocesses the files. It calculates annual model means for 1981–2010, interpolates to five fixed depths, then regrids temperature and salinity to 1°. WOA's 1981–2010 normal is already an annual climatology and is not averaged again. The supplied Gadi file has invalid time bounds, which the notebook discards after loading; the representative time point remains unchanged. WOA in-situ temperature is converted to potential temperature using WOA salinity. The paired mask requires both variables in the model and reference.

In [ ]:
from dask.distributed import Client
from esmvalcore.dataset import Dataset
from esmvalcore.preprocessor import annual_statistics, extract_levels, regrid

try:
    worker_count = max(1, int(os.environ.get('PBS_NCPUS', '1')))
except ValueError:
    worker_count = 1
owns_client = False
try:
    client = Client.current()
except ValueError:
    try:
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError):
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)
    owns_client = True
active_workers = len(client.scheduler_info()['workers'])
print(f'Dask distributed: {active_workers} workers '
      f'(requested {worker_count}); dashboard: {client.dashboard_link}')

def prepared(dataset, climatology=False):
    cube = dataset.load()
    if climatology:
        time = cube.coord('time')
        if time.has_bounds() and not np.all(
                (time.bounds[:, 0] <= time.points)
                & (time.points <= time.bounds[:, 1])):
            print('WOA time bounds do not contain its representative '
                  'timestamp; ignoring invalid bounds')
            time.bounds = None
    else:
        cube = annual_statistics(cube, operator='mean')
    cube = extract_levels(cube, levels=LEVELS, scheme='linear')
    return regrid(cube, target_grid=TARGET_GRID, scheme='linear')

In [ ]:
common = dict(project='CMIP6', dataset=MODEL, exp='historical',
              ensemble=ENSEMBLE, grid=GRID, mip='Omon',
              timerange=HISTORICAL_YEARS)
model_cube = prepared(Dataset(short_name='thetao', **common))
model_s_cube = prepared(Dataset(short_name='so', **common))
reference_t_cube = prepared(Dataset(
    project='OBS6', dataset='WOA', type='clim', version=WOA_VERSION,
    tier=2, mip='Omon', short_name='thetao', timerange=WOA_TIMERANGE),
    climatology=True)
reference_s_cube = prepared(Dataset(
    project='OBS6', dataset='WOA', type='clim', version=WOA_VERSION,
    tier=2, mip='Omon', short_name='so', timerange=WOA_TIMERANGE),
    climatology=True)
model_cube.convert_units('degC')
model_s_cube.convert_units('1e-3')
_compatible(model_cube, model_s_cube)
_compatible(model_cube, reference_t_cube)
model_t_series, depth, years = _annual_fields(model_cube)
model_s_series, _, salinity_years = _annual_fields(model_s_cube)
_check_reference_period(years)
if not np.array_equal(years, salinity_years):
    raise ValueError('temperature and salinity years differ')
model_t = np.ma.mean(model_t_series, axis=0)
model_s = np.ma.mean(model_s_series, axis=0)
reference_t = np.ma.mean(_woa_potential_temperature(
    reference_t_cube, reference_s_cube), axis=0)
reference_s_cube.convert_units('1e-3')
reference_s = np.ma.mean(_annual_fields(reference_s_cube)[0], axis=0)
lat2d, lon2d = lat_2d(model_cube), lon_2d(model_cube)

reference_valid = (~np.ma.getmaskarray(reference_t)
                   & ~np.ma.getmaskarray(reference_s)
                   & np.isfinite(np.ma.filled(reference_t, np.nan))
                   & np.isfinite(np.ma.filled(reference_s, np.nan)))
thermal, haline, total = density_components(
    model_t, model_s, reference_t, reference_s, depth, lat2d, lon2d)
metrics = density_metrics(thermal, haline, total, reference_valid,
                          _region_weights(model_cube))
closure = np.ma.max(np.ma.abs(total - thermal - haline))
if closure > 1e-9:
    raise AssertionError(f'density attribution does not close: {closure}')
print(f'Maximum decomposition residual: {closure:.2e} kg m-3')
print('Global paired coverage at 10 m:',
      f"{metrics['coverage'][0, 0]:.1%}")

## 1. Where do temperature and salinity reinforce or cancel?

The first two columns give each variable's contribution to the **model minus WOA** in-situ density at the stated depth. Their sum is the third column. Positive means denser model water. At a fixed depth, the same pressure is used for both states. The colour scale is shared across the three terms at each depth, so the cancellation is visible rather than hidden by rescaling.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8),
                         constrained_layout=True, sharex=True, sharey=True)
for row, level_index in enumerate((0, 3)):
    fields = (thermal[level_index], haline[level_index], total[level_index])
    vmax = float(np.nanpercentile(np.abs(np.ma.filled(fields, np.nan)), 98))
    vmax = max(vmax, 1e-3)
    for col, (field, label) in enumerate(zip(
            fields, ('Temperature effect', 'Salinity effect', 'Net density'))):
        ax = axes[row, col]
        image = ax.pcolormesh(model_cube.coord('longitude').points,
                              model_cube.coord('latitude').points, field,
                              cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                              shading='auto', rasterized=True)
        ax.set_title(f'{label} at {depth[level_index]:g} m')
        ax.set_xlabel('Longitude (°E)')
        ax.set_ylabel('Latitude (°N)')
        fig.colorbar(image, ax=ax, label='kg m$^{-3}$')
fig.suptitle(f'{MODEL} — density attribution')
show_figure(fig)
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / 'density_attribution_maps.png', dpi=160)
plt.close(fig)

## 2. How does compensation vary with depth?

The regional profiles use cell-area weights on the paired wet mask. The cancellation fraction is the fraction of the *absolute* temperature and salinity density effects that offsets locally, weighted by their magnitude and cell area. Zero means reinforcement; one means exact cancellation. It is not a measure of observational confidence. Check paired coverage alongside deep results.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5),
                         constrained_layout=True, sharey=True)
for region, (label, _, _) in enumerate(REGIONS):
    axes[0].plot(metrics['total'][region], depth, marker='o', label=label)
    axes[1].plot(metrics['cancellation'][region], depth,
                 marker='o', label=label)
axes[0].axvline(0., color='0.3', lw=0.8)
axes[0].set_xlabel('Net density difference (kg m$^{-3}$)')
axes[1].set_xlabel('Local cancellation fraction (0–1)')
axes[1].set_xlim(0, 1)
axes[0].set_ylabel('Depth (m)')
axes[0].invert_yaxis()
axes[0].legend(fontsize=8)
fig.suptitle(MODEL)
show_figure(fig)
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / 'density_compensation_profiles.png', dpi=160)
plt.close(fig)
print('Global net density difference (kg m-3):',
      np.round(metrics['total'][0], 4))
print('Global paired coverage:', np.round(metrics['coverage'][0], 3))

## Interpretation and cleanup

An apparently small net density difference can hide sizeable, opposing temperature and salinity differences. The exact decomposition shows which term is responsible and the cancellation fraction quantifies the offset. WOA coverage describes where the atlas has gridded values; it does not describe confidence, especially at depth. For an ACCESS-OM3 application, verify whether a specific output file contains conservative or potential temperature and absolute or practical salinity before using these equations.

In [ ]:
if owns_client:
    client.close()
    print('Notebook-owned Dask client closed')
elif client is not None:
    print('Existing Dask client left open')